# Part 2: Baseline model

In [1]:
import sys
sys.path.append('..')

import pandas as pd
from src.baseline_features import (
    build_text_features,
    combine_with_metadata,
    train_logreg,
    evaluate_model
)
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings('ignore', category=ConvergenceWarning)

In [2]:
# Part 2 - Data paths 
TRAIN_PATH = '../data/processed/news_stratified_train.csv'
VAL_PATH = '../data/processed/news_stratified_val.csv'

# Load content for optional metadata features if available
cols_to_keep = ['content_processed', 'content', 'type']

train_df = pd.read_csv(TRAIN_PATH, low_memory=False, usecols=cols_to_keep, on_bad_lines='skip')
val_df = pd.read_csv(VAL_PATH, low_memory=False, usecols=cols_to_keep, on_bad_lines='skip')

# Basic cleaning
train_df = train_df.dropna(subset=['content_processed', 'type']).copy()
val_df = val_df.dropna(subset=['content_processed', 'type']).copy()

train_df['type'] = train_df['type'].astype(int)
val_df['type'] = val_df['type'].astype(int)

print(f"Train: {train_df.shape}, Val: {val_df.shape}")
print(f"\nTrain label distribution:\n{train_df['type'].value_counts()}")
print(f"\nVal label distribution:\n{val_df['type'].value_counts()}")


Train: (757768, 3), Val: (94721, 3)

Train label distribution:
type
1    427304
0    330464
Name: count, dtype: int64

Val label distribution:
type
1    53413
0    41308
Name: count, dtype: int64


We load the stratified train and validation splits from the full 995K dataset. The splits follow an 80/10/10 ratio with preserved class balance across labels.

## Feature Extraction
We use CountVectorizer with the top 10,000 most frequent words as text features. We also build a metadata feature matrix with URL count, number count, and article length for comparison in Task 3.

In [3]:
# Part 2 - Build feature matrices
X_train, X_val, vectorizer = build_text_features(
    train_df['content_processed'],
    val_df['content_processed'],
    max_features=10000
)

y_train = train_df['type'].values
y_val = val_df['type'].values

print(f"X_train shape: {X_train.shape}")
print(f"X_val shape: {X_val.shape}")

X_train shape: (757768, 10000)
X_val shape: (94721, 10000)


In [4]:
# Optional metadata matrices for Task 3
X_train_meta = combine_with_metadata(X_train, train_df)
X_val_meta = combine_with_metadata(X_val, val_df)

print(f"X_train + metadata shape: {X_train_meta.shape}")
print(f"X_val + metadata shape: {X_val_meta.shape}")

X_train + metadata shape: (757768, 10003)
X_val + metadata shape: (94721, 10003)


## Logistic Regression Baseline
We train a logistic regression classifier on text features only. We set class_weight='balanced' to handle class imbalance in the training data.

In [5]:
# Part 2 - Train logistic regression baseline
model = train_logreg(
    X_train,
    y_train,
    C=1.0,
    max_iter=100,
    solver='liblinear',
    class_weight='balanced'
)

print("Model trained.")
print(f"Hyperparameters: C={model.C}, max_iter={model.max_iter}, solver={model.solver}, class_weight={model.class_weight}")

Model trained.
Hyperparameters: C=1.0, max_iter=100, solver=liblinear, class_weight=balanced


## Evaluation on Validation Set
We evaluate the model on the validation set using macro F1-score and a full classification report.

In [6]:
# Part 2 - Evaluate baseline on validation set
results = evaluate_model(model, X_val, y_val, label_names=['reliable', 'fake'])

print(results['report'])
print(f"Macro F1: {results['macro_f1']:.4f}")
print(f"Binary F1: {results['binary_f1']:.4f}")
print("\nConfusion matrix:")
print(results['confusion_matrix'])

              precision    recall  f1-score   support

    reliable       0.85      0.83      0.84     41308
        fake       0.87      0.89      0.88     53413

    accuracy                           0.86     94721
   macro avg       0.86      0.86      0.86     94721
weighted avg       0.86      0.86      0.86     94721

Macro F1: 0.8576
Binary F1: 0.8776

Confusion matrix:
[[34081  7227]
 [ 5995 47418]]


## Summary
The baseline model achieves a macro F1 of 0.8579 on the validation set using text features only. Fake articles are classified slightly better (F1=0.88) than reliable articles (F1=0.84).

## Save Model
We save the trained model and vectorizer for use in evaluation.

In [7]:
import joblib

joblib.dump(model, '../models/baseline_model.joblib')
joblib.dump(vectorizer, '../models/baseline_vectorizer.joblib')

print("Model and vectorizer saved.")

Model and vectorizer saved.
